# Video Frame Interpolation

Uses **FILM (Frame Interpolation for Large Motion)** from TensorFlow Hub.
Self-contained — no broken imports, no manual patching.

**First:** `Runtime > Change runtime type > T4 GPU`

In [ ]:
!nvidia-smi

In [ ]:
# Install everything needed
!pip install -q tensorflow tensorflow-hub opencv-python-headless einops
!apt-get install -q ffmpeg
print('Done!')

In [ ]:
import os, cv2, shutil, math, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files as colab_files
import tensorflow as tf
import tensorflow_hub as hub

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Load FILM model from TF Hub (downloads ~100MB once)
print('Loading FILM model...')
model = hub.load('https://tfhub.dev/google/film/1')
print('FILM model loaded!')

## Upload, Interpolate, Download

Set `FPS_OUT` to your target frame rate and upload your video.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
FPS_OUT = 60   # <-- desired output fps (e.g. 60 for 10fps->60fps)
# ─────────────────────────────────────────────────────────────────────────────

# ── Step 1: Upload and sanitize filename ─────────────────────────────────────
print('Upload your video...')
uploaded = colab_files.upload()
orig = list(uploaded.keys())[0]
safe = '/content/input_raw.mp4'
os.rename(f'/content/{orig}', safe)
print(f'Saved as: {safe}  ({os.path.getsize(safe)/1e6:.1f} MB)')

# ── Step 2: Remux to ensure compatibility ────────────────────────────────────
os.makedirs('/content/videos', exist_ok=True)
INPUT_VIDEO = '/content/videos/input.mp4'
os.system(f'ffmpeg -y -i {safe} -c:v libx264 -preset fast -pix_fmt yuv420p -an {INPUT_VIDEO} -loglevel warning')
print(f'Remuxed: {os.path.getsize(INPUT_VIDEO)/1e6:.1f} MB')

# ── Step 3: Get video info ────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=width,height,r_frame_rate',
     '-of', 'default=noprint_wrappers=1', INPUT_VIDEO],
    capture_output=True, text=True
)
W, H, FPS_IN = 0, 0, 0.0
for line in result.stdout.splitlines():
    if 'width='  in line: W = int(line.split('=')[1])
    if 'height=' in line: H = int(line.split('=')[1])
    if 'r_frame_rate' in line:
        n, d = line.split('=')[1].strip().split('/')
        FPS_IN = float(n) / float(d)
print(f'Input: {W}x{H} @ {FPS_IN:.2f} fps')
assert W > 0 and FPS_IN > 0, 'Could not read video'

# ── Step 4: Extract frames ───────────────────────────────────────────────────
shutil.rmtree('/content/frames_in',  ignore_errors=True)
shutil.rmtree('/content/frames_out', ignore_errors=True)
os.makedirs('/content/frames_in')
os.makedirs('/content/frames_out')

os.system(f'ffmpeg -y -i {INPUT_VIDEO} -vsync 0 /content/frames_in/frame_%05d.png -loglevel error')
src = sorted(Path('/content/frames_in').glob('*.png'))
print(f'Extracted {len(src)} frames')
assert len(src) > 1, 'No frames extracted'

# ── Step 5: Preview ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
fig.suptitle(f'Input — {FPS_IN:.1f} fps')
for ax, i in zip(axes, [0, len(src)//3, 2*len(src)//3, -1]):
    img = cv2.cvtColor(cv2.imread(str(src[i])), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(f'Frame {i}'); ax.axis('off')
plt.tight_layout(); plt.show()

# ── Step 6: Interpolate with FILM ────────────────────────────────────────────
MULTIPLIER     = max(2, 2 ** round(math.log2(FPS_OUT / FPS_IN)))
FPS_ACTUAL_OUT = FPS_IN * MULTIPLIER
print(f'Target: {FPS_ACTUAL_OUT:.1f} fps  (x{MULTIPLIER})')

def read_frame(path):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0

def film_interpolate(model, f0, f1, num_frames):
    """Insert num_frames-1 frames between f0 and f1 using FILM."""
    results = [f0]
    for t in range(1, num_frames):
        ts = t / num_frames
        inp = {
            'x0': tf.expand_dims(tf.cast(f0, tf.float32), 0),
            'x1': tf.expand_dims(tf.cast(f1, tf.float32), 0),
            'time': tf.reshape(tf.cast(ts, tf.float32), [1, 1]),
        }
        mid = model(inp)['image'][0].numpy()
        results.append(mid)
    return results

out_idx = 0
print(f'Interpolating {len(src)} frames...')
for i in range(len(src) - 1):
    f0 = read_frame(src[i])
    f1 = read_frame(src[i + 1])
    frames = film_interpolate(model, f0, f1, MULTIPLIER)
    for frm in frames:
        out = (np.clip(frm, 0, 1) * 255).astype(np.uint8)
        cv2.imwrite(f'/content/frames_out/frame_{out_idx:05d}.png',
                    cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
        out_idx += 1
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(src)-1} pairs done...')

# Write last frame
last = (np.clip(read_frame(src[-1]), 0, 1) * 255).astype(np.uint8)
cv2.imwrite(f'/content/frames_out/frame_{out_idx:05d}.png',
            cv2.cvtColor(last, cv2.COLOR_RGB2BGR))
print(f'Done! {out_idx+1} output frames')

# ── Step 7: Encode ───────────────────────────────────────────────────────────
OUTPUT_VIDEO = '/content/videos/output_interpolated.mp4'
os.system(
    f'ffmpeg -y -framerate {int(FPS_ACTUAL_OUT)}'
    f' -i /content/frames_out/frame_%05d.png'
    f' -c:v libx264 -pix_fmt yuv420p -crf 18'
    f' {OUTPUT_VIDEO} -loglevel error'
)
mb = os.path.getsize(OUTPUT_VIDEO) / 1e6
print(f'Saved: {OUTPUT_VIDEO} ({mb:.1f} MB)')
print(f'{FPS_IN:.1f} fps -> {FPS_ACTUAL_OUT:.1f} fps ({MULTIPLIER}x)')

# ── Step 8: Download ─────────────────────────────────────────────────────────
colab_files.download(OUTPUT_VIDEO)

## Optional: Quality Check (PSNR)

In [ ]:
src_files = sorted(Path('/content/frames_in').glob('*.png'))
out_files = sorted(Path('/content/frames_out').glob('*.png'))

if len(src_files) > 2:
    a = cv2.imread(str(src_files[1])).astype(np.float64)
    b = cv2.imread(str(out_files[1])).astype(np.float64)
    if a.shape != b.shape:
        b = cv2.resize(b, (a.shape[1], a.shape[0]))
    mse = np.mean((a - b) ** 2)
    score = 20 * np.log10(255 / np.sqrt(mse)) if mse > 0 else float('inf')
    print(f'PSNR: {score:.2f} dB  (TOFlow baseline ~33.5 dB | higher = better)')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'PSNR: {score:.2f} dB')
    axes[0].imshow(cv2.cvtColor(cv2.imread(str(src_files[1])), cv2.COLOR_BGR2RGB))
    axes[0].set_title('Ground Truth'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(cv2.imread(str(out_files[1])), cv2.COLOR_BGR2RGB))
    axes[1].set_title('Interpolated'); axes[1].axis('off')
    plt.tight_layout(); plt.show()